<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/geospatial/HW_4/Week04_Interpolation_Kriging_Kwanda_1077167.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import libraries**

In [ ]:
!pip install -q geostatspy statsmodels geostatsmodels numba

In [ ]:
import os                                                     # for file and directory operations
import numpy as np                                            # for numerical operations and ndarrays
import pandas as pd                                           # for handling data in DataFrame format
import matplotlib.pyplot as plt                               # for plotting and visualization
from statsmodels.stats.weightstats import DescrStatsW         # for weighted statistics calculations
from scipy.optimize import curve_fit                          # for curve fitting
import geostatspy.GSLIB as GSLIB                              # GSLIB utilities, visualization, and wrapper functions
import geostatspy.geostats as geostats                        # GSLIB methods converted to Python
import geostatspy                                             # for accessing the package version and general utilities


from tqdm import tqdm                                         # progress bar for loops
from functools import partialmethod                           # used to modify tqdm to suppress the status bar
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)    # suppress the status bar for tqdm

from matplotlib.ticker import (MultipleLocator, AutoMinorLocator) # control over axes ticks in plots
plt.rc('axes', axisbelow=True)                                # ensure grid lines are plotted below plot elements

import scipy.spatial as sp                                    # for spatial data structures and algorithms (KDTree)
import geopandas as gpd                                       # for handling geospatial data (GeoDataFrame)
from matplotlib import gridspec                               # custom subplots

# Define ignore_warnings before using it
ignore_warnings = True                                        # flag to ignore warnings

# Check and apply the warning filter
if ignore_warnings == True:
    import warnings
    warnings.filterwarnings('ignore')

from IPython.utils import io                                  # mute output from simulation

cmap = "Spectral_r"                                           # color map


# **Functions**

In [ ]:
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator) # control of axes ticks

def add_grid():
    ax = plt.gca()
    ax.grid(True, which='major', linewidth=1.0, alpha=0.7)
    ax.grid(True, which='minor', linewidth=0.5, linestyle='--', alpha=0.4)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=4)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())

# **1. Data & Week 3 Foundations**

### 1.1 Load Data

In [ ]:
# Load data from Github
url = "https://raw.githubusercontent.com/kwanda2426/projects/main/geospatial/HW_4/CuMineBH.csv"
df = pd.read_csv(url)
df_cu = df.copy()
df_cu.head()

##### Summary statistics

In [ ]:
df_cu.describe().T

##### Scatter Map

In [ ]:
# Create the figure and axis
fig, ax = plt.subplots(figsize = (12, 6))
# Create the scatter plot
sc = ax.scatter(df_cu['EAST'],df_cu['NORTH'], s = 10,c = df_cu['Cu'], cmap = cmap)
# Create the grid
ax.grid(True, color = 'grey', linestyle = '--', linewidth = 0.5)

# Add labels to the axes
ax.set_xlabel('EAST')
ax.set_ylabel('NORTH')
ax.set_title('Map of Point Locations Coloured by Cu%')

# Add a colorbar
cbar = plt.colorbar(sc, ax = ax)
cbar.set_label('Cu (ppb)', rotation = 90, labelpad = 0)
cbar.ax.tick_params(labelsize=10)
add_grid()

# Display the plot
plt.show()

### 1.2 Declustering recap

In [ ]:
# Adding declustered weights
df_cu["declust_weight"], cell_size, declust_means = geostats.declus(
    df_cu, 'EAST', 'NORTH', 'Cu',
    iminmax = 1, noff = 100, ncell = 200, cmin = 1, cmax = 500)


# Calculate naive and declustered means
weighted_data = DescrStatsW(df_cu["Cu"].values, weights = df_cu["declust_weight"].values, ddof = 0)
dmean = round(weighted_data.mean, 3)
naive_mean = round(np.mean(df_cu["Cu"].values), 3)
# storing declustered mean
mu_declust = dmean
df_cu['mu_declust'] = round(weighted_data.mean, 3)

print("Naive mean =", naive_mean)
print("Declustered mean =", dmean)

In [ ]:
# Identify best declustered cell size (min of declust_means)
min_dmean_idx = np.argmin(declust_means)
best_cell_size = cell_size[min_dmean_idx]
min_declust_val = declust_means[min_dmean_idx]

# Create the plot
plt.figure(figsize=(14, 6))
plt.scatter(cell_size, declust_means, s = 30, alpha = 0.8,
            edgecolors = "black", facecolors = 'goldenrod', label = 'Declustered Mean')

# Horizontal lines
plt.axhline(y = naive_mean, color = 'darkblue', linestyle = '-', label = 'Naive Mean')
plt.axhline(y = dmean, color = 'green', linestyle = '-', label = 'Declustered Mean')

# Vertical line at minimum declustered mean
plt.axvline(x = best_cell_size, color = 'red', linestyle = '--',
            label = f'Min Declust Mean @ Cell Size = {best_cell_size:.0f}')

# Labels, title, grid, legend
plt.xlabel('Cell Size (m)')
plt.ylabel('Declustered Mean (Cu %)')
plt.title('Declustered Mean vs. Cell Size for Cu')
plt.grid(color = 'grey', linestyle = '-.', linewidth = 0.3, which = 'both')
plt.legend()

# Show the plot
plt.show()

In [ ]:
# Create the figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Create the scatter plot
sc = ax.scatter(df_cu['EAST'], df_cu['NORTH'], s = 10, c = df_cu['declust_weight'], cmap = 'Spectral_r')

# Add labels to the axes
ax.set_xlabel('EAST')
ax.set_ylabel('NORTH')
ax.set_title('Map of Point Locations Coloured by Declustered weights')

# Add a colorbar
cbar = plt.colorbar(sc, ax = ax)
cbar.set_label('Cell Declustering Weights', rotation = 270, labelpad = 15)
cbar.ax.tick_params(labelsize=10)
add_grid()

# Display the plot
plt.show()


In [ ]:
# Calculate naive statistics
naive_stats = df_cu['Cu'].describe(percentiles=[0.25, 0.75])

# Calculate declustered statistics using weighted statistics
declust_mean = weighted_data.mean
declust_min = np.min(df_cu['Cu'].values)
declust_q1 = weighted_data.quantile(0.25).item()  # Convert to scalar using item()
declust_q3 = weighted_data.quantile(0.75).item()  # Convert to scalar using item()
declust_max = np.max(df_cu['Cu'].values)

# Construct the table data
table_data = {
    "Copper Data": ["Naive", "Declustered", "Difference"],
    "Min [ppb]": [round(naive_stats['min'], 2), round(declust_min, 2), round(declust_min - naive_stats['min'], 2)],
    "Q1 [ppb]": [round(naive_stats['25%'], 2), round(declust_q1, 2), round(declust_q1 - naive_stats['25%'], 2)],
    "Mean [ppb]": [round(naive_stats['mean'], 2), round(declust_mean, 2), round(declust_mean - naive_stats['mean'], 2)],
    "Q3 [ppb]": [round(naive_stats['75%'], 2), round(declust_q3, 2), round(declust_q3 - naive_stats['75%'], 2)],
    "Max [ppb]": [round(naive_stats['max'], 2), round(declust_max, 2), round(declust_max - naive_stats['max'], 2)]
}

# Create a DataFrame for the table
summary_table_df = pd.DataFrame(table_data)
summary_table_df


### 1.3 Normal score transform (NST) recap

In [ ]:
df_cu["nscore"], tv, tns = geostats.nscore(df_cu, "Cu", "declust_weight")
df_cu.head()

In [ ]:
# Extracting features
feature = "Cu"
feature_units = "ppb"
vmin = df_cu["Cu"].min()
vmax = df_cu["Cu"].max()
xmin = df_cu["EAST"].min()
xmax = df_cu["EAST"].max()
ymin = df_cu["NORTH"].min()
ymax = df_cu["NORTH"].max()

In [ ]:
plt.subplot(221)
GSLIB.hist_st(df_cu['Cu'],0,30,log = False,cumul = False,bins = 25,weights = df_cu['declust_weight'],xlabel = "Cu (ppb)",title = "Declustered Cu")
plt.ylim(0.0,1800)
add_grid()
plt.subplot(222)
GSLIB.hist_st(df_cu['nscore'],-3.0,3.0,log = False,cumul = False,bins = 25,weights = df_cu['declust_weight'],xlabel = "Normal Scores Cu",title = "Gaussian Transformed Cu")
plt.ylim(0.0,180)
add_grid()
plt.subplot(223)
GSLIB.locmap_st(df_cu,'EAST','NORTH','Cu',xmin,xmax,ymin,ymax,vmin,vmax,'Declustered Cu','EAST','NORTH','Cu (ppb)',cmap)
add_grid()
plt.subplot(224)
GSLIB.locmap_st(df_cu,'EAST','NORTH','nscore',xmin,xmax,ymin,ymax,-3.0,3.0,'Gaussian Transformed Cu','EAST','NORTH','Normal Scores Cu',cmap)
add_grid()
plt.subplots_adjust(left = 0.0, bottom = 0.0, right = 2.0, top = 2.1, wspace = 0.2, hspace = 0.2)
plt.show()

In [ ]:
plt.subplot(121)

plt.hist(df_cu[feature], facecolor = 'red',bins=np.linspace(vmin,vmax,1000),histtype = "stepfilled",alpha = 0.2,density = True,cumulative = True,edgecolor = 'black')
plt.xlim([vmin,vmax]); plt.ylim([0,1.0])
plt.xlabel(feature + '(' + feature_units + ')'); plt.ylabel('Frequency'); plt.title('Cu')
add_grid()

plt.subplot(122)
plt.hist(df_cu['nscore'], facecolor = 'blue',bins = np.linspace(-3.0,3.0,1000),histtype = "stepfilled",alpha = 0.2,density = True,cumulative = True,edgecolor = 'black')
plt.xlim([-3.0,3.0]); plt.ylim([0,1.0])
plt.xlabel('Gaussian Transformed ' + feature); plt.ylabel('Frequency'); plt.title('Guassian Transformed ' + feature)
add_grid()

plt.subplots_adjust(left = 0.0, bottom = 0.0, right = 2.0, top = 1.2, wspace = 0.2, hspace = 0.3)
output_file = 'Cu_CuNScore_CDF.png'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

### 1.4 Variogram recap & model selection

In [ ]:
# ───────────────────────────────────────────────
# Variogram Parameter Setup (GSLIB/GeoStatSpy style)
# ───────────────────────────────────────────────

tmin = -99999         # Minimum value threshold (no trimming applied here)
tmax = 99999          # Maximum value threshold (no trimming applied here)

lag_dist = 8         # Lag spacing (distance between successive lags in variogram)
lag_tol = lag_dist*0.5  # Lag tolerance (half lag distance allows ±3.5 units tolerance)

nlag = 25            # Number of lag bins to compute the variogram over

bandh = 9999.9        # Bandwidth (across-strike tolerance); very large value disables band restriction (omnidirectional)

atol = 15           # Azimuth tolerance (in degrees); allows matching directions within ±22.5° of specified azimuth

isill = 1             # Sill standardisation (1 = experimental variogram is normalised to sill = 1)

azi_mat = [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165]
                     # Azimuth directions (in degrees) to compute directional variograms
                     # Note: there’s a duplication of 135 – check for typo or intentional redundancy

xlimax = lag_dist * nlag  # Maximum distance on x-axis for variogram plot (controls display range)

min_samples = 3      # Minimum number of pairs required per lag to compute a valid semivariance


In [ ]:
# Arrays to store the results
lag = np.zeros((len(azi_mat),nlag+2)); gamma = np.zeros((len(azi_mat),nlag+2)); npp = np.zeros((len(azi_mat),nlag+2));

for iazi in range(0,len(azi_mat)):                      # Loop over all directions
    lag[iazi,:], gamma[iazi,:], npp[iazi,:] = geostats.gamv(df_cu,"EAST","NORTH","nscore",tmin,tmax,lag_dist,lag_tol,nlag,azi_mat[iazi],atol,bandh,isill)
    plt.subplot(6,2,iazi+1)  # Changed subplot layout to 4 rows and 3 columns
    plt.plot(lag[iazi,:],gamma[iazi,:],'o',color = 'black',label = 'Azimuth ' +str(azi_mat[iazi]))
    plt.plot([0,2000],[1.0,1.0],color = 'black')
    plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
    plt.ylabel(r'$\gamma \bf(h)$')
    plt.title('Directional NSCORE As Variogram')
    plt.xlim([0,300])
    plt.ylim([0,1.8])
    plt.legend(loc='upper left')
    add_grid()

plt.subplots_adjust(left=0.0, bottom=0.0, right=2.0, top=4.2, wspace=0.2, hspace=0.3)
plt.show()

- We can observe that Azimuth 000 is the major direction and Azimuth 090 is the minor direction. at 000 the sill is not reached, hence it is the maximum distance, and it follows that the orthogonal direction to that is 090.

- This corresponds to imajor = 0 and iminor = 6

In [ ]:
# Select the plot above for the major and minor
imajor = 0
iminor = 6

print('Major direction is ' + str(azi_mat[imajor]) + ' azimuth.')
print('Minor direction is ' + str(azi_mat[iminor]) + ' azimuth.')

if not abs(azi_mat[imajor] - azi_mat[iminor]) == 90.0:
    print('Major and minor directions must be orthogonal to each other.')
    sys.exit()

plt.subplot(1,2,1)
plt.plot(lag[imajor,:],gamma[imajor,:],'o',color = 'grey',label = 'Azimuth ' + str(azi_mat[imajor]))
plt.plot([0,xlimax],[1.0,1.0],color = 'black')
plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
plt.ylabel(r'$\gamma \bf(h)$')
plt.title('Directional NSCORE Cu Variogram - Major ' + str(azi_mat[imajor]) + ' Azimuth')
plt.xlim([0,xlimax])
plt.ylim([0,1.4])
plt.legend(loc='upper left')
add_grid()
plt.subplot(1,2,2)
plt.plot(lag[iminor,:],gamma[iminor,:],'o',color = 'gold',label = 'Azimuth ' +str(azi_mat[iminor]))
plt.plot([0,xlimax],[1.0,1.0],color = 'black')
plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
plt.ylabel(r'$\gamma \bf(h)$')
plt.title('Directional NSCORE As Variogram  - Minor ' + str(azi_mat[iminor]) + ' Azimuth')
plt.xlim([0,xlimax])
plt.ylim([0,1.4])
plt.legend(loc='upper left')
add_grid()
plt.subplots_adjust(left=0.0, bottom=0.0, right=2.0, top=1.1, wspace=0.2, hspace=0.3); plt.show()

#### Spherical fit

In [ ]:
nug = 0.30; nst = 2                                             # 2 nest structure variogram model parameters
it1 = 1; cc1 = 0.20; azi1 = azi_mat[imajor]; hmaj1 = 125; hmin1 = 110
it2 = 1; cc2 = (1-(nug + cc1)); azi2 = azi_mat[imajor]; hmaj2 = 400; hmin2 = 190

vario = GSLIB.make_variogram(nug,nst,it1,cc1,azi1,hmaj1,hmin1,it2,cc2,azi2,hmaj2,hmin2) # make model object
nlag = 25; xlag = 10;                                          # project the model in the 045 azimuth
index_maj,h_maj,gam_maj,cov_maj,ro_maj = geostats.vmodel(nlag,xlag,azi_mat[imajor],vario)                                                     # project the model in the 135 azimuth
index_min,h_min,gam_min,cov_min,ro_min = geostats.vmodel(nlag,xlag,azi_mat[iminor],vario)

plt.subplot(1,2,1)
plt.scatter(lag[imajor,:],gamma[imajor,:],color = 'grey',edgecolor = 'black',marker = 'o',label = 'Azimuth ' +str(azi_mat[imajor]))
plt.plot([0,xlimax],[1.0,1.0],color = 'black',linestyle = '--'  )
plt.plot(h_maj,gam_maj,color = 'grey',lw = 3,zorder = 60, label = 'Major Spherical Model Fit')
plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
plt.ylabel(r'$\gamma \bf(h)$')
plt.title('Directional NSCORE Cu Variogram - Major ' + str(azi_mat[imajor]) + ' Azimuth')
plt.xlim([0,xlimax])
plt.ylim([0,1.4])
plt.legend(loc = 'upper left')
add_grid()

plt.subplot(1,2,2)
plt.scatter(lag[iminor,:],gamma[iminor,:],color = 'gold',edgecolor = 'black', marker = 'o',label = 'Azimuth ' +str(azi_mat[iminor]))
plt.plot([0,xlimax],[1.0,1.0],color = 'black',linestyle = '--' ) # sill line
plt.plot(h_min,gam_min,color = 'gold',linewidth = 3,zorder = 60, label = 'Minor Spherical Model Fit')
plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
plt.ylabel(r'$\gamma \bf(h)$')
plt.title('Directional NSCORE Cu Variogram  - Minor ' + str(azi_mat[iminor]) + ' Azimuth')
plt.xlim([0,xlimax])
plt.ylim([0,1.4])
plt.legend(loc = 'upper left')
add_grid()
plt.subplots_adjust(left = 0.0, bottom = 0.0, right = 2.0, top = 1.1, wspace = 0.2, hspace = 0.3); plt.show()

#### Exponential fit

In [ ]:
nug = 0.30; nst = 2                                             # 2 nest structure variogram model parameters
it1 = 2; cc1 = 0.20; azi1 = azi_mat[imajor]; hmaj1 = 125; hmin1 = 110
it2 = 2; cc2 = (1-(nug + cc1)); azi2 = azi_mat[imajor]; hmaj2 = 400; hmin2 = 190

vario = GSLIB.make_variogram(nug,nst,it1,cc1,azi1,hmaj1,hmin1,it2,cc2,azi2,hmaj2,hmin2) # make model object
nlag = 25; xlag = 10;                                          # project the model in the 045 azimuth
index_maj,h_maj,gam_maj,cov_maj,ro_maj = geostats.vmodel(nlag,xlag,azi_mat[imajor],vario)                                                     # project the model in the 135 azimuth
index_min,h_min,gam_min,cov_min,ro_min = geostats.vmodel(nlag,xlag,azi_mat[iminor],vario)

plt.subplot(1,2,1)
plt.scatter(lag[imajor,:],gamma[imajor,:],color = 'grey',edgecolor = 'black',marker = 'o',label = 'Azimuth ' +str(azi_mat[imajor]))
plt.plot([0,xlimax],[1.0,1.0],color = 'black',linestyle = '--'  )
plt.plot(h_maj,gam_maj,color = 'grey',lw = 3,zorder = 60, label = 'Major Exponential Model Fit')
plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
plt.ylabel(r'$\gamma \bf(h)$')
plt.title('Directional NSCORE Cu Variogram - Major ' + str(azi_mat[imajor]) + ' Azimuth')
plt.xlim([0,xlimax])
plt.ylim([0,1.4])
plt.legend(loc = 'upper left')
add_grid()

plt.subplot(1,2,2)
plt.scatter(lag[iminor,:],gamma[iminor,:],color = 'gold',edgecolor = 'black', marker = 'o',label = 'Azimuth ' +str(azi_mat[iminor]))
plt.plot([0,xlimax],[1.0,1.0],color = 'black',linestyle = '--' ) # sill line
plt.plot(h_min,gam_min,color = 'gold',linewidth = 3,zorder = 60, label = 'Minor Exponential Model Fit')
plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
plt.ylabel(r'$\gamma \bf(h)$')
plt.title('Directional NSCORE Cu Variogram  - Minor ' + str(azi_mat[iminor]) + ' Azimuth')
plt.xlim([0,xlimax])
plt.ylim([0,1.4])
plt.legend(loc = 'upper left')
add_grid()
plt.subplots_adjust(left = 0.0, bottom = 0.0, right = 2.0, top = 1.1, wspace = 0.2, hspace = 0.3); plt.show()

#### Gaussian Fit

In [ ]:
nug = 0.30; nst = 2                                             # 2 nest structure variogram model parameters
it1 = 3; cc1 = 0.30; azi1 = azi_mat[imajor]; hmaj1 = 125; hmin1 = 110
it2 = 3; cc2 = (1-(nug + cc1)); azi2 = azi_mat[imajor]; hmaj2 = 400; hmin2 = 190

vario = GSLIB.make_variogram(nug,nst,it1,cc1,azi1,hmaj1,hmin1,it2,cc2,azi2,hmaj2,hmin2) # make model object
nlag = 25; xlag = 10;                                          # project the model in the 045 azimuth
index_maj,h_maj,gam_maj,cov_maj,ro_maj = geostats.vmodel(nlag,xlag,azi_mat[imajor],vario)                                                     # project the model in the 135 azimuth
index_min,h_min,gam_min,cov_min,ro_min = geostats.vmodel(nlag,xlag,azi_mat[iminor],vario)

plt.subplot(1,2,1)
plt.scatter(lag[imajor,:],gamma[imajor,:],color = 'grey',edgecolor = 'black',marker = 'o',label = 'Azimuth ' +str(azi_mat[imajor]))
plt.plot([0,xlimax],[1.0,1.0],color = 'black',linestyle = '--'  )
plt.plot(h_maj,gam_maj,color = 'grey',lw = 3,zorder = 60, label = 'Major Gaussian Model Fit')
plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
plt.ylabel(r'$\gamma \bf(h)$')
plt.title('Directional NSCORE Cu Variogram - Major ' + str(azi_mat[imajor]) + ' Azimuth')
plt.xlim([0,xlimax])
plt.ylim([0,1.4])
plt.legend(loc = 'upper left')
add_grid()

plt.subplot(1,2,2)
plt.scatter(lag[iminor,:],gamma[iminor,:],color = 'gold',edgecolor = 'black', marker = 'o',label = 'Azimuth ' +str(azi_mat[iminor]))
plt.plot([0,xlimax],[1.0,1.0],color = 'black',linestyle = '--' ) # sill line
plt.plot(h_min,gam_min,color = 'gold',linewidth = 3,zorder = 60, label = 'Minor Gaussian Model Fit')
plt.xlabel(r'Lag Distance $\bf(h)$, (m)')
plt.ylabel(r'$\gamma \bf(h)$')
plt.title('Directional NSCORE Cu Variogram  - Minor ' + str(azi_mat[iminor]) + ' Azimuth')
plt.xlim([0,xlimax])
plt.ylim([0,1.4])
plt.legend(loc = 'upper left')
add_grid()
plt.subplots_adjust(left = 0.0, bottom = 0.0, right = 2.0, top = 1.1, wspace = 0.2, hspace = 0.3); plt.show()

# **2. Interpolation Grid & Search Strategy**

In [ ]:
# Parameters
idw_radius = int(hmaj2)  # major range in metres
idw_power_list = [1,2,3,4,5] # Power will change
ndmin = 3
ndmax = 20
xsiz = 10 #block size for estimation
ysiz = 10 #block size for estimation
nx = int(np.ceil((xmax-xmin)/xsiz)) #number of blocks along a diretion
ny = int(np.ceil((ymax-ymin)/ysiz)) #number of blocks along a diretion
xmn = xmin + xsiz * 0.5
ymn = ymin + ysiz * 0.5

ivtype = 1                                                   # variable type, 0 - categorical, 1 - continuous
as_skmean = declust_mean                                      # declustered

radius = int(hmaj2)                                           # search radius for neighbouring data
nxdis = 1; nydis = 1                                          # number of grid discretizations for block kriging


as_vario = GSLIB.make_variogram(nug,nst,it1,cc1,azi1,hmaj1,hmin1,it2,cc2,azi2,hmaj2,hmin2) # make model object

sk_ktype = 0                                                     # kriging type, 0 - simple, 1 - ordinary
ok_ktype = 1                                                     # kriging type, 0 - simple, 1 - ordinary

📍 Spatial Estimation Parameters
idw_radius = int(hmaj2)
• 	Uses the major range from the variogram model as the search radius.
• 	Ensures that only spatially correlated points are used in IDW interpolation.
idw_power = 2
• 	Standard choice for IDW: weights decrease with the square of distance.
• 	Balances influence between nearby and farther points.
ndmin = 3, ndmax = 20
• 	Controls the number of neighboring data points used.
• 	Ensures stability (minimum of 3) and avoids overfitting (maximum of 20).

🧱 Grid and Block Setup
xsiz = 10, ysiz = 10
• 	Defines the block size for estimation.
• 	Smaller blocks give finer resolution but increase computation.
nx, ny
• 	Number of blocks along X and Y directions.
• 	Calculated to fully cover the spatial extent of your data.
xmn = xmin + xsiz * 0.5, ymn = ymin + ysiz * 0.5
• 	Sets the origin of the grid at the center of the first block.
• 	Ensures alignment of estimation grid with data extent.

📊 Variable and Mean Settings
ivtype = 1
• 	Indicates a continuous variable, suitable for kriging and IDW.
as_skmean = declust_mean
• 	Uses a declustered mean for Simple Kriging, reducing bias from clustered data.

🔍 Search and Discretization
radius = int(hmaj2)
• 	Again uses the variogram’s major range to define the search radius for kriging.
• 	Ensures spatial relevance of neighboring data.
nxdis = 1; nydis = 1
• 	Minimal discretization for block kriging.
• 	Suitable for small blocks or when computational efficiency is needed.

📐 Variogram and Kriging Type
as_vario = GSLIB.make_variogram(...)
• 	Constructs a variogram model using nugget, sill, range, and anisotropy.
• 	Essential for kriging to model spatial correlation.
sk_ktype = 0, ok_ktype = 1
• 	Specifies Simple Kriging (SK) and Ordinary Kriging (OK).
• 	SK uses a known mean (declustered), OK estimates the mean locally.

# **3. IDW Interpolation (p = 1, 2, 3, 4, 5)**

### 3.1 Method

In [ ]:
# Inverse Distance Weighting
def invdist(
    df,
    xcol,
    ycol,
    vcol,
    tmin,
    tmax,
    nx,
    xmn,
    xsiz,
    ny,
    ymn,
    ysiz,
    ndmin,
    ndmax,
    radius,
    power
    ):
    """ Based on modification of the GSLIB kb2d program by Deutsch and Journel (1997)
    :param df: pandas DataFrame with the spatial data
    :param xcol: name of the x coordinate column
    :param ycol: name of the y coordinate column
    :param vcol: name of the property column
    :param tmin: property trimming limit
    :param tmax: property trimming limit
    :param nx: definition of the grid system (x axis)
    :param xmn: definition of the grid system (x axis)
    :param xsiz: definition of the grid system (x axis)
    :param ny: definition of the grid system (y axis)
    :param ymn: definition of the grid system (y axis)
    :param ysiz: definition of the grid system (y axis)
    :param ndmin: minimum number of data points to use for kriging a block
    :param ndmax: maximum number of data points to use for kriging a block
    :param radius: maximum isotropic search radius
    :param power: the inverse distance power
    :return: estmap (estimated map)
    """

    # Constants
    UNEST = -999.
    EPSLON = 1.0e-10

    # Load the data
    df_extract = df.loc[(df[vcol] >= tmin) & (df[vcol] <= tmax)]    # trim values outside tmin and tmax
    nd = len(df_extract)
    ndmax = min(ndmax,nd)
    x = df_extract[xcol].values
    y = df_extract[ycol].values
    vr = df_extract[vcol].values

    # Allocate the needed memory:
    xa = np.zeros(ndmax)
    ya = np.zeros(ndmax)
    vra = np.zeros(ndmax)
    dist = np.zeros(ndmax)
    nums = np.zeros(ndmax)
    s = np.zeros(ndmax)
    estmap = np.full((ny, nx), UNEST)  # Estimation map initialized with UNEST

    # Make a KDTree for fast search of nearest neighbours
    data_locs = np.column_stack((y, x))
    tree = sp.cKDTree(data_locs, leafsize = 16, compact_nodes = True, copy_data = False, balanced_tree = True)

    # MAIN LOOP OVER ALL THE BLOCKS IN THE GRID:
    for iy in tqdm(range(ny)):
        yloc = ymn + iy * ysiz
        for ix in range(nx):
            xloc = xmn + ix * xsiz
            current_node = (yloc, xloc)

            # Find the nearest samples within the search radius
            dist, nums = tree.query(current_node, ndmax) # use kd tree for fast nearest data search
            nums = nums[dist < radius]
            dist = dist[dist < radius]
            nd = len(dist)

            # Is there enough samples?
            if nd < ndmin:
                est  = UNEST
            else:
                # Put coordinates and values of neighborhood samples into xa, ya, vra:
                for ia in range(nd):
                    jj = int(nums[ia])
                    xa[ia]  = x[jj]
                    ya[ia]  = y[jj]
                    vra[ia] = vr[jj]

                # Solve for weights
                dist = np.sqrt((xa[:nd]-xloc)**2 + (ya[:nd]-yloc)**2)
                s = 1 / ((dist + EPSLON)**power)        # calculate inverse weights
                s = s / np.sum(s)                       # constrain sum of the weights to 1.0 for unbiasedness

                est = np.sum(s * vra[:nd])  # Calculate the estimate

            estmap[iy, ix] = est

    return estmap

### 3.2 Maps

In [ ]:
for idw_power in idw_power_list:
    # Run IDW
    invdist_map_cu = invdist(df_cu, 'EAST', 'NORTH', 'Cu', tmin, tmax, nx, xmn, xsiz, ny, ymn, ysiz, ndmin, ndmax, idw_radius, idw_power)

    # Prepare meshgrid
    xx, yy = np.meshgrid(np.arange(xmn, xmn + nx * xsiz, xsiz), np.arange(ymn, ymn + ny * ysiz, ysiz))
    idw_map_cu = invdist_map_cu.flatten().copy()

    # Create DataFrame
    idw_est = pd.DataFrame({'EAST': xx.flatten(), 'NORTH': yy.flatten(), 'IDW_cu': idw_map_cu})
    idw_est_gdf = gpd.GeoDataFrame(idw_est, geometry=gpd.points_from_xy(idw_est.EAST, idw_est.NORTH))

    # Adjust coordinates
    idw_est_gdf["EAST"] += xsiz / 2
    idw_est_gdf["NORTH"] += ysiz / 2
    idw_est_gdf = idw_est_gdf.reset_index(drop=True)

    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    imx = ax.scatter(idw_est_gdf["EAST"], idw_est_gdf["NORTH"], c=idw_est_gdf["IDW_cu"],
                     marker="s", s=10, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.scatter(df_cu['EAST'], df_cu['NORTH'], c=df_cu['Cu'], s=20, marker="o",
               vmin=vmin, vmax=vmax, cmap=cmap, alpha=1, edgecolors='k')
    ax.axis('scaled')
    cbar = fig.colorbar(imx, ax=ax, fraction=0.048, pad=0.00)
    cbar.set_label(f'IDW{int(idw_power)} EST* Cu (ppm)')
    ax.set_title(f'IDW{int(idw_power)} EST* Cu (ppm)')
    ax.set_xlabel('EAST')
    ax.set_ylabel('NORTH')
    add_grid()
    plt.show()


### 3.3 Observation summary

# **4. Simple Kriging (SK)**

### 4.1 Assumptions & setup

In [ ]:
df_cu.head()

In [ ]:
# Using mu_declust
# mu_declust
sk_out_lst = []
sk_cu_kmap, sk_cu_vmap = geostats.kb2d(df_cu,'EAST','NORTH','mu_declust',vmin,vmax,nx,xmn,xsiz,ny,ymn,ysiz,nxdis,nydis,
         ndmin,ndmax,radius,sk_ktype,as_skmean,as_vario)

sk_kmap = np.clip(sk_cu_kmap, 0, vmax)
sk_out_lst.append([sk_kmap, sk_cu_vmap])

### 4.2 Estimation

In [ ]:
# Prepare meshgrid for mapping kriged values to coordinates
xx, yy = np.meshgrid(np.arange(xmin, xmax, xsiz), np.arange(ymin, ymax, ysiz))

# Flatten the kriged maps and combine them into a DataFrame
sk_kmap_cu = np.flipud(sk_out_lst[0][0]).flatten().copy()
sk_vmap_cu = abs(np.flipud(sk_out_lst[0][1]).flatten().copy())

sk_est = pd.DataFrame(list(zip(xx.flatten(), yy.flatten(), sk_kmap_cu, sk_vmap_cu)), columns = ['EAST', 'NORTH', 'SK_cu', 'SK_KVAR'])

# Convert to GeoDataFrame
sk_est_gdf = gpd.GeoDataFrame(sk_est, geometry=gpd.points_from_xy(sk_est.EAST, sk_est.NORTH))


# Adjust 'X' and 'Y' coordinates to their original positions (centers of cells)
sk_est_gdf["EAST"] = sk_est_gdf["EAST"] + xsiz / 2
sk_est_gdf["NORTH"] = sk_est_gdf["NORTH"] + ysiz / 2
sk_est_gdf = sk_est_gdf.reset_index(drop=True)

sk_est_gdf.head()

### 4.3 Maps

In [ ]:
#plot the data using matplotlib
fig, ax = plt.subplots(figsize = (14, 6))
# Scatter plot
imx = ax.scatter(sk_est_gdf["EAST"] , sk_est_gdf["NORTH"],c = sk_est_gdf["SK_cu"], marker = "s", s=20, vmin = vmin,vmax = vmax,cmap = cmap)
plt.scatter(df_cu['EAST'],df_cu['NORTH'], c= df_cu['Cu'],
            s = 20, marker = "o",vmin = vmin, vmax = vmax,cmap = cmap, alpha = 1, edgecolors = 'k')
# Set axis properties
ax.axis('scaled')

# Add a color bar with more control
cbar = fig.colorbar(imx, ax = ax, fraction = 0.048, pad = 0.00)  # Corrected to use 'scatter' for the color bar
cbar.set_label('SK EST* As (ppb)')
ax.set_title('SK EST* As (ppb)')
# Add axis labels and a title
ax.set_xlabel('EAST')
ax.set_ylabel('NORTH')
add_grid()
plt.show()


#plot the data using matplotlib
fig, ax = plt.subplots(figsize = (14, 6))
# Scatter plot
imx = ax.scatter(sk_est_gdf["EAST"] , sk_est_gdf["NORTH"],c=sk_est_gdf["SK_KVAR"], marker = "s", s=20, vmin = vmin,vmax = vmax,cmap = cmap)
plt.scatter(df_cu['EAST'],df_cu['NORTH'], color = 'grey',
            s = 20, marker = "o")
# Set axis properties
ax.axis('scaled')

# Add a color bar with more control
cbar = fig.colorbar(imx, ax = ax, fraction = 0.048, pad = 0.00)  # Corrected to use 'scatter' for the color bar
cbar.set_label('SK EST* As (ppb)')
ax.set_title('SK KVAR* As (ppb)')
# Add axis labels and a title
ax.set_xlabel('EAST')
ax.set_ylabel('NORTH')
add_grid()
plt.show()

# **5. Ordinary Kriging (OK)**

### 5.1 Assumptions & setup

### 5.2 Estimation

### 5.3 Maps

# **6. For each method, compute Validation & Comparative Analysis**

# **7. Observations and Conclusion**